<a href="https://colab.research.google.com/github/luizalmin-netizen/js-developer-pokedex/blob/main/Jogo_da_Forca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Estrutura de Classes para o Jogo da Forca em Python

Vamos definir algumas classes principais para organizar a lógica do jogo.

In [1]:
import random

class WordManager:
    """
    Gerencia a palavra secreta, as letras adivinhadas e a lógica de verificação.
    """
    def __init__(self, word_list):
        self.secret_word = random.choice(word_list).upper()
        self.guessed_letters = set()
        self.revealed_word = ['_'] * len(self.secret_word)

    def guess_letter(self, letter):
        letter = letter.upper()
        if letter in self.guessed_letters:
            return False, "Você já adivinhou essa letra."

        self.guessed_letters.add(letter)

        if letter in self.secret_word:
            for i, char in enumerate(self.secret_word):
                if char == letter:
                    self.revealed_word[i] = letter
            return True, f"Boa! '{letter}' está na palavra."
        else:
            return False, f"Pena! '{letter}' não está na palavra."

    def get_display_word(self):
        return ' '.join(self.revealed_word)

    def is_word_guessed(self):
        return '_' not in self.revealed_word

    def get_secret_word(self):
        return self.secret_word


### Classe `HangmanDisplay`

Esta classe será responsável por toda a interação visual com o usuário, como exibir a forca, a palavra oculta e as letras já tentadas. Isso ajuda a manter a lógica do jogo separada da apresentação.

In [7]:
class HangmanDisplay:
    """
    Responsável por exibir o estado do jogo para o usuário.
    """
    HANGMAN_PICS = [
        """
   -----
   |   |
   |
   |
   |
   |
---------
        """,
        """
   -----
   |   |
   |   O
   |
   |
   |
---------
        """,
        """
   -----
   |   |
   |   O
   |   |
   |
   |
---------
        """,
        """
   -----
   |   |
   |   O
   |  /|
   |
   |
---------
        """,
        """
   -----
   |   |
   |   O
   |  /|\
   |
   |
---------
        """,
        """
   -----
   |   |
   |   O
   |  /|\
   |  /
   |
---------
        """,
        """
   -----
   |   |
   |   O
   |  /|\
   |  / \
   |
---------
        """
    ]

    def display_game_state(self, display_word, guessed_letters, remaining_attempts):
        print(self.HANGMAN_PICS[6 - remaining_attempts])
        print(f"Palavra: {display_word}")
        print(f"Letras tentadas: {', '.join(sorted(list(guessed_letters)))}")
        print(f"Tentativas restantes: {remaining_attempts}")
        print("-" * 30)

    def get_player_guess(self):
        while True:
            guess = input("Adivinhe uma letra: ").strip().upper()
            if len(guess) == 1 and guess.isalpha():
                return guess
            else:
                print("Entrada inválida. Por favor, digite uma única letra.")

    def show_message(self, message):
        print(message)

    def show_final_message(self, won, secret_word):
        if won:
            print("Parabéns! Você adivinhou a palavra!")
        else:
            print("Game Over! Você ficou sem tentativas.")
        print(f"A palavra secreta era: {secret_word}")

    def get_play_again_choice(self):
        while True:
            choice = input("Quer jogar novamente? (S/N): ").strip().upper()
            if choice == 'S':
                return True
            elif choice == 'N':
                return False
            else:
                print("Escolha inválida. Por favor, digite 'S' para sim ou 'N' para não.")

    def get_initial_menu_choice(self):
        while True:
            print("\n--- Menu Principal ---")
            print("1. Novo Jogo")
            print("2. Carregar Jogo Salvo")
            print("3. Sair")
            choice = input("Escolha uma opção: ").strip()
            if choice in ['1', '2', '3']:
                return choice
            else:
                print("Opção inválida. Por favor, escolha 1, 2 ou 3.")

    def get_save_game_choice(self):
        while True:
            choice = input("Deseja salvar o jogo? (S/N): ").strip().upper()
            if choice == 'S':
                return True
            elif choice == 'N':
                return False
            else:
                print("Escolha inválida. Por favor, digite 'S' para sim ou 'N' para não.")


### Classe `HangmanGame`

Esta é a classe principal que orquestra o jogo. Ela conterá a lógica para iniciar o jogo, gerenciar o loop de adivinhação, verificar vitórias/derrotas e lidar com o estado geral do jogo.

In [8]:
import json
import os # Para verificar a existência do arquivo

class HangmanGame:
    """
    Orquestra o fluxo principal do jogo da forca.
    """
    def __init__(self, word_list, max_attempts=6):
        self.word_list = word_list # Armazena a lista de palavras para resetar o jogo
        self.display = HangmanDisplay()
        self.max_attempts = max_attempts
        self.word_manager = None # Inicializado em reset_game ou load_game
        self.remaining_attempts = max_attempts
        self.game_over = False

    def reset_game(self):
        self.word_manager = WordManager(self.word_list)
        self.remaining_attempts = self.max_attempts
        self.game_over = False

    def save_game(self, filename='hangman_save.json'):
        game_state = {
            'secret_word': self.word_manager.get_secret_word(),
            'guessed_letters': list(self.word_manager.guessed_letters), # Sets não são serializáveis em JSON
            'remaining_attempts': self.remaining_attempts
        }
        with open(filename, 'w') as f:
            json.dump(game_state, f)
        self.display.show_message(f"Jogo salvo com sucesso em '{filename}'!")

    def load_game(self, filename='hangman_save.json'):
        if not os.path.exists(filename):
            self.display.show_message("Nenhum jogo salvo encontrado.")
            return False

        try:
            with open(filename, 'r') as f:
                game_state = json.load(f)

            # Reconstruir o estado do jogo
            loaded_secret_word = game_state['secret_word']
            loaded_guessed_letters = set(game_state['guessed_letters'])
            loaded_remaining_attempts = game_state['remaining_attempts']

            # Criar um novo WordManager com a palavra secreta carregada
            # e, em seguida, definir as letras adivinhadas e a palavra revelada
            self.word_manager = WordManager([loaded_secret_word]) # Passa a palavra carregada como uma lista
            self.word_manager.secret_word = loaded_secret_word
            self.word_manager.guessed_letters = loaded_guessed_letters

            # Reconstruir revealed_word
            self.word_manager.revealed_word = ['_'] * len(self.word_manager.secret_word)
            for letter in loaded_guessed_letters:
                if letter in self.word_manager.secret_word:
                    for i, char in enumerate(self.word_manager.secret_word):
                        if char == letter:
                            self.word_manager.revealed_word[i] = letter

            self.remaining_attempts = loaded_remaining_attempts
            self.game_over = False
            self.display.show_message("Jogo carregado com sucesso!")
            return True
        except (FileNotFoundError, json.JSONDecodeError, KeyError) as e:
            self.display.show_message(f"Erro ao carregar o jogo: {e}")
            return False

    def start_game(self):
        while True: # Loop para permitir múltiplas rodadas ou ações do menu
            choice = self.display.get_initial_menu_choice()

            if choice == '1': # Novo Jogo
                self.reset_game()
                self.display.show_message("Iniciando um novo jogo!")
            elif choice == '2': # Carregar Jogo Salvo
                if not self.load_game():
                    continue # Volta para o menu se o carregamento falhar
            elif choice == '3': # Sair
                self.display.show_message("Obrigado por jogar!")
                break

            # Inicia o loop do jogo após a escolha do menu (novo ou carregado)
            if choice in ['1', '2']:
                while not self.game_over:
                    self._game_loop()

                # Jogo terminou (ganhou/perdeu), perguntar se quer jogar novamente
                if self.display.get_save_game_choice():
                    self.save_game()

                play_again = self.display.get_play_again_choice()
                if not play_again:
                    self.display.show_message("Obrigado por jogar!")
                    break # Sai do loop principal se o usuário não quiser jogar novamente
                else:
                    self.reset_game() # Reseta o jogo para a próxima rodada (voltará ao menu inicial)


    def _game_loop(self):
        self.display.display_game_state(
            self.word_manager.get_display_word(),
            self.word_manager.guessed_letters,
            self.remaining_attempts
        )

        if self.word_manager.is_word_guessed():
            self.game_over = True
            self.display.show_final_message(True, self.word_manager.get_secret_word())
            return

        if self.remaining_attempts <= 0:
            self.game_over = True
            self.display.show_final_message(False, self.word_manager.get_secret_word())
            return

        guess = self.display.get_player_guess()
        is_correct, message = self.word_manager.guess_letter(guess)
        self.display.show_message(message)

        if not is_correct:
            self.remaining_attempts -= 1


# Exemplo de uso:
if __name__ == '__main__':
    words = ["python", "programacao", "desafio", "computador", "algoritmo"]
    game = HangmanGame(words)
    game.start_game()



--- Menu Principal ---
1. Novo Jogo
2. Carregar Jogo Salvo
3. Sair


KeyboardInterrupt: Interrupted by user

## Testes Unitários para `WordManager`

Vamos criar testes unitários para garantir que a lógica de adivinhação da classe `WordManager` esteja funcionando conforme o esperado. Utilizaremos o módulo `unittest` do Python.

In [4]:
import unittest

# A classe WordManager já foi definida anteriormente, mas precisamos garantir que ela está disponível
# para o ambiente de teste. Se você estiver executando este código em um ambiente que limpa o estado,
# você pode precisar redefinir a classe aqui ou garantir que ela foi executada antes.


class TestWordManager(unittest.TestCase):

    def setUp(self):
        # Configura um novo WordManager para cada teste
        self.word_list = ["TESTE"]
        self.manager = WordManager(self.word_list)

    def test_initial_state(self):
        self.assertEqual(self.manager.secret_word, "TESTE")
        self.assertEqual(self.manager.get_display_word(), "_ _ _ _ _")
        self.assertFalse(self.manager.is_word_guessed())

    def test_correct_guess(self):
        is_correct, message = self.manager.guess_letter('T')
        self.assertTrue(is_correct)
        self.assertEqual(self.manager.get_display_word(), "T _ _ T _")
        self.assertIn('T', self.manager.guessed_letters)
        self.assertFalse(self.manager.is_word_guessed())

    def test_incorrect_guess(self):
        is_correct, message = self.manager.guess_letter('X')
        self.assertFalse(is_correct)
        self.assertEqual(self.manager.get_display_word(), "_ _ _ _ _")
        self.assertIn('X', self.manager.guessed_letters)
        self.assertFalse(self.manager.is_word_guessed())

    def test_repeated_guess(self):
        self.manager.guess_letter('T') # First guess
        is_correct, message = self.manager.guess_letter('T') # Repeated guess
        self.assertFalse(is_correct)
        self.assertEqual(message, "Você já adivinhou essa letra.")
        self.assertEqual(self.manager.get_display_word(), "T _ _ T _")

    def test_word_guessed(self):
        self.manager.guess_letter('T')
        self.manager.guess_letter('E')
        self.manager.guess_letter('S')
        self.assertEqual(self.manager.get_display_word(), "T E S T E")
        self.assertTrue(self.manager.is_word_guessed())

    def test_mixed_guesses(self):
        self.manager.guess_letter('A') # Incorrect
        self.manager.guess_letter('T') # Correct
        self.manager.guess_letter('E') # Correct
        self.assertEqual(self.manager.get_display_word(), "T E _ T E")
        self.assertIn('A', self.manager.guessed_letters)
        self.assertIn('T', self.manager.guessed_letters)
        self.assertIn('E', self.manager.guessed_letters)
        self.assertFalse(self.manager.is_word_guessed())


In [ ]:
# Para rodar os testes:
# O unittest.main() geralmente é usado quando o arquivo é executado diretamente.
# No Jupyter/Colab, é melhor usar unittest.TextTestRunner para um controle mais limpo.

if __name__ == '__main__':
    # Isso cria uma suíte de testes a partir da classe TestWordManager
    suite = unittest.TestSuite()
    suite.addTest(unittest.makeSuite(TestWordManager))

    # Isso executa os testes na suíte
    runner = unittest.TextTestRunner(verbosity=2)
    runner.run(suite)


### Resumo da Estrutura:

*   **`WordManager`**: Encapsula tudo relacionado à palavra secreta, incluindo o estado das letras adivinhadas e a lógica para verificar tentativas.
*   **`HangmanDisplay`**: Lida exclusivamente com a interface do usuário (entrada e saída), tornando o jogo mais fácil de adaptar a diferentes UIs (console, GUI, web, etc.).
*   **`HangmanGame`**: Atua como o controlador principal, coordenando as ações entre `WordManager` e `HangmanDisplay` e gerenciando o fluxo e o estado do jogo.

Esta estrutura promove a **separação de responsabilidades** e o **encapsulamento**, tornando o código mais modular, testável e fácil de manter.

## Como Documentar o Projeto no `README.md` para o GitHub

Um `README.md` bem estruturado é crucial para qualquer projeto no GitHub. Ele serve como a porta de entrada para seu projeto, explicando o que ele é, como funciona e como utilizá-lo. Abaixo está um template que você pode adaptar para o seu jogo da forca, cobrindo os pontos essenciais:

---

```markdown
# Jogo da Forca em Python (POO)

## Visão Geral

Este projeto implementa o clássico jogo da forca em ambiente de console, utilizando os princípios da Programação Orientada a Objetos (POO) em Python. Desenvolvido como parte do desafio "Entendendo o Desafio" da DIO, o objetivo principal foi aplicar e demonstrar a compreensão de conceitos como classes, objetos, encapsulamento e separação de responsabilidades.

## Objetivos de Aprendizagem Demonstrados

Durante o desenvolvimento deste jogo, foram aplicados os seguintes conceitos de POO e programação:

*   **Classes e Objetos**: Definição de classes como `WordManager`, `HangmanDisplay` e `HangmanGame` para representar diferentes entidades e suas responsabilidades.
*   **Encapsulamento**: Proteção dos dados internos de cada classe, expondo apenas métodos públicos para interação.
*   **Separação de Responsabilidades**: Cada classe tem um único propósito bem definido (gerenciar a palavra, exibir o estado do jogo, orquestrar o fluxo do jogo).
*   **Métodos**: Implementação de comportamentos específicos para cada objeto.
*   **Lógica de Jogo**: Controle de estado do jogo (tentativas restantes, letras adivinhadas, etc.) e fluxo (início, loop de jogo, fim).
*   **Manipulação de Strings**: Para exibir a palavra oculta e formatar mensagens.
*   **Entrada e Saída do Usuário**: Interação com o jogador via console.
*   **Testes Unitários**: Garantia da correção da lógica principal do jogo.

## Funcionalidades

*   Seleção aleatória de uma palavra secreta de uma lista predefinida.
*   Exibição gráfica simples da forca, evoluindo com cada erro.
*   Visualização da palavra com as letras adivinhadas e lacunas para as letras desconhecidas.
*   Acompanhamento das letras já tentadas.
*   Controle de tentativas restantes.
*   Mensagens informativas para o jogador sobre o status do jogo.
*   Funcionalidade de jogar novamente após o término de uma rodada.

## Estrutura do Projeto

O projeto é modularizado em três classes principais:

*   **`WordManager`**:
    *   **Responsabilidade**: Gerenciar a palavra secreta, as letras já adivinhadas e a lógica de verificação de tentativas.
    *   **Métodos Chave**: `__init__`, `guess_letter`, `get_display_word`, `is_word_guessed`, `get_secret_word`.

*   **`HangmanDisplay`**:
    *   **Responsabilidade**: Lidar com toda a interação visual e textual com o usuário (entrada e saída).
    *   **Métodos Chave**: `display_game_state`, `get_player_guess`, `show_message`, `show_final_message`, `get_play_again_choice`.
    *   **Recursos**: Contém as representações ASCII da forca (`HANGMAN_PICS`).

*   **`HangmanGame`**:
    *   **Responsabilidade**: Orquestrar o fluxo principal do jogo, coordenando as outras duas classes e gerenciando o estado geral do jogo.
    *   **Métodos Chave**: `__init__`, `start_game`, `_game_loop`, `reset_game`.

Esta arquitetura promove a alta coesão e baixo acoplamento entre os componentes, facilitando a manutenção e futuras extensões do jogo.

## Como Executar o Jogo

Para rodar o jogo em seu ambiente local, siga os passos abaixo:

1.  **Clone o repositório:**
    ```bash
    git clone https://github.com/seu-usuario/seu-repositorio-da-forca.git
    cd seu-repositorio-da-forca
    ```
2.  **Execute o arquivo principal:**
    ```bash
    python main_game.py # Ou o nome do seu arquivo principal, e.g., hangman.py
    ```
    *(Assumindo que o código das classes e o bloco `if __name__ == '__main__':` estão no mesmo arquivo ou devidamente importados.)*

## Executando os Testes Unitários

O projeto inclui testes unitários para a classe `WordManager` para garantir a lógica de adivinhação. Para executá-los:

1.  **Certifique-se de que o arquivo de testes (e.g., `test_word_manager.py`) esteja no diretório correto.**
2.  **Execute o módulo `unittest`:**
    ```bash
    python -m unittest seu_arquivo_de_testes.py # e.g., python -m unittest test_word_manager.py
    ```
    Ou se você incluiu o bloco `if __name__ == '__main__':` para execução no mesmo arquivo:
    ```bash
    python seu_arquivo_principal.py
    ```
    Você verá a saída dos testes, indicando se passaram ou falharam.

## Contribuição

Sinta-se à vontade para propor melhorias, corrigir bugs ou adicionar novas funcionalidades. Para contribuir:

1.  Faça um fork do projeto.
2.  Crie uma nova branch (`git checkout -b feature/AmazingFeature`).
3.  Faça suas mudanças e comite (`git commit -m 'Add some AmazingFeature'`).
4.  Envie para a branch (`git push origin feature/AmazingFeature`).
5.  Abra um Pull Request.

## Licença

Distribuído sob a licença MIT. Veja `LICENSE` para mais informações.

---

## Como Incluir Capturas de Tela no Repositório GitHub

Adicionar capturas de tela ao seu `README.md` é uma excelente maneira de visualizar o seu projeto e demonstrar seu funcionamento sem que o usuário precise executar o código. Siga os passos abaixo:

### 1. Capturar as Telas do Jogo

No ambiente Google Colab:

*   **Execute as células do jogo da forca** e interaja com ele para alcançar diferentes estados (início do jogo, uma letra adivinhada, alguns erros, vitória, derrota).
*   **Tire capturas de tela** do console de saída do Colab (onde o jogo está sendo executado) para cada estado que você deseja mostrar. A maioria dos sistemas operacionais tem ferramentas nativas para isso:
    *   **Windows**: Ferramenta de Captura (Snipping Tool) ou `Print Screen` (`PrtSc`).
    *   **macOS**: `Shift + Command + 4` para selecionar uma área.
    *   **Linux**: Ferramentas como `Spectacle` (KDE) ou `gnome-screenshot` (GNOME).
*   **Salve as imagens** em um formato comum, como `.png` ou `.jpg`, com nomes descritivos (ex: `inicio_jogo.png`, `erro_forca.png`, `vitoria.png`).

### 2. Organizar as Imagens no Repositório GitHub

É uma boa prática criar uma pasta específica para suas imagens no repositório:

1.  No seu repositório GitHub (ou localmente antes de fazer o push):
    *   Crie uma pasta chamada `images` (ou `screenshots`, `assets`, etc.) na raiz do seu projeto.
    *   Mova todas as suas capturas de tela salvas para dentro desta pasta.

    ```
    seu-repositorio/
    ├── .git/
    ├── images/
    │   ├── inicio_jogo.png
    │   ├── erro_forca.png
    │   └── vitoria.png
    ├── main_game.py
    ├── README.md
    └── ...
    ```

2.  Faça o commit e o push dessas imagens para o seu repositório GitHub.

### 3. Inserir as Imagens no `README.md`

No seu arquivo `README.md`, você pode linkar as imagens usando a sintaxe Markdown para imagens:

```markdown
## Demonstração do Jogo

Abaixo estão algumas capturas de tela do jogo da forca em ação:

### Início do Jogo
![Estado inicial do jogo da forca](images/inicio_jogo.png)
*A palavra oculta e 6 tentativas restantes.*

### Jogo em Andamento (com erros)
![Exemplo de jogo com 3 erros](images/erro_forca.png)
*O boneco da forca começa a ser desenhado após os erros.*

### Vitória!
![Tela de vitória do jogo](images/vitoria.png)
*Parabéns! A palavra foi adivinhada com sucesso.*

### Derrota (Game Over)
![Tela de derrota do jogo](images/game_over.png)
*Você ficou sem tentativas. A palavra secreta é revelada.*
```

**Pontos Importantes:**

*   **`alt text` (texto alternativo)**: `[Estado inicial do jogo da forca]` é importante para acessibilidade e é exibido se a imagem não carregar. Seja descritivo.
*   **Caminho da Imagem**: `(images/inicio_jogo.png)` deve ser o caminho **relativo** correto da sua imagem a partir do `README.md`.
*   **Descrições**: Adicione breves descrições (`*A palavra oculta...*`) abaixo de cada imagem para dar contexto.

Ao seguir esses passos, seu `README.md` ficará muito mais profissional e informativo, oferecendo uma ótima visão do seu projeto aos visitantes do seu repositório!